# triangle-barycentric composite — cx22: unbind a batched (N, 3) solve result into three (N,) tensors (s, u, v)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `triangle-barycentric`, `tensor-unbind`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "triangle-barycentric"
DD_ATOM_IDS = ["triangle-barycentric", "tensor-unbind"]
DD_SUBTOPICS = ["Geometry: Barycentric coords", "Numpy: Indexing and selection"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

After solving the barycentric 3x3 system in batch — `sol = t.linalg.solve(M, y)` with shape `(N, 3)` — you want the three SCALAR-PER-RAY component tensors `s`, `u`, `v` separately so you can build the inside-test mask: `(u >= 0) & (v >= 0) & (u + v <= 1)`.

`tensor.unbind(dim=-1)` does this in one call: it returns a tuple of `dim`-removed tensors, each `(N,)`. Compared to `sol[:, 0], sol[:, 1], sol[:, 2]` it's a single named call and reads as 'split this axis into its constituent components' — which is the load-bearing semantic.

### Composite Exercise — unbind a batched (N, 3) solve result into three (N,) tensors (s, u, v)

**Atoms exercised together**: `triangle-barycentric`, `tensor-unbind`

Implement `cx22_unbind_suv(sol)` that takes the `(N, 3)` solve output and returns three `(N,)` tensors `(s, u, v)`.

Use `sol.unbind(dim=-1)` (or `t.unbind(sol, dim=-1)`) and return the resulting tuple.

The test checks: tuple length 3, each component shape `(N,)`, exact-equality with `sol[:, 0]`, `sol[:, 1]`, `sol[:, 2]`, and that the components share storage with `sol` (unbind produces views, not copies).

In [ ]:
def cx22_unbind_suv(sol):
    # Atom A (triangle-barycentric): the last axis of sol is (s, u, v).
    # Atom B (tensor-unbind): split that axis into three (N,) views in one call.
    return sol.unbind(dim=-1)


<details><summary>Show solution — cx22</summary>

```python
def cx22_unbind_suv(sol):
    # Atom A (triangle-barycentric): the last axis of sol is (s, u, v).
    # Atom B (tensor-unbind): split that axis into three (N,) views in one call.
    return sol.unbind(dim=-1)
```

`unbind(dim=-1)` is the named alternative to indexing each column manually — for a 3-component barycentric result it's three lines compressed to one, and the returned tensors are views (stride magic, not copies). This composes nicely with the inside-test in cx24 where each component flows into its own predicate.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx22'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx22',
        'subtopics': ["Geometry: Barycentric coords", "Numpy: Indexing and selection"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()